# 02 - Exploratory Data AnalysisPresentation surface over `src/analytics/`. Every figure below is the *same file*`scripts/run_eda.py` writes to `reports/figures/` - the notebook displays artefacts,it does not recompute them differently.Prerequisite:```bashpython scripts/run_eda.py```

In [ ]:
import sys, jsonfrom pathlib import PathROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))import pandas as pdfrom src.utils.config import load_config, resolve_pathfrom src.utils.logging import setup_loggingsetup_logging("WARNING")          # keep the notebook readablepd.set_option("display.width", 160)pd.set_option("display.max_columns", 40)data_config = load_config("data_config")PROCESSED = resolve_path(data_config.require("paths.processed"))INTERIM   = resolve_path(data_config.require("paths.interim"))TABLES    = resolve_path(data_config.require("paths.tables"))FIGURES   = resolve_path(data_config.require("paths.figures"))print("tier:", data_config.require("active_tier"))

## 1. Headline figures

In [ ]:
summary = json.loads((TABLES / "eda_summary.json").read_text())pd.Series(summary["summary"]).to_frame("value")

## 2. What the data is telling usThese notes are generated from the data, not written by hand.

In [ ]:
for i, note in enumerate(summary["notes"], 1):    print(f"{i}. {note}\n")

## 3. Users

In [ ]:
from IPython.display import ImageImage(str(FIGURES / "user_distributions.png"))

## 4. Items: the long tailThe Gini coefficient sets the bar. A strong popularity baseline is expected, and any model that cannot beat it is not learning personalisation.

In [ ]:
Image(str(FIGURES / "long_tail.png"))

## 5. Reviews

In [ ]:
Image(str(FIGURES / "review_distributions.png"))

## 6. TimeVolume, rating trend and active users. The measured per-user category drift in the notes above is the empirical premise for temporal weighting (Module 13).

In [ ]:
Image(str(FIGURES / "temporal_trends.png"))

In [ ]:
Image(str(FIGURES / "category_trends.png"))

## 7. Graph structure

In [ ]:
graph_stats = json.loads((TABLES / "graph_stats.json").read_text())display(pd.Series(graph_stats["metrics"]).to_frame("value"))print("\nApproximated metrics (exact computation is O(V*E)):")display(pd.DataFrame(graph_stats["approximations"]).T)

In [ ]:
Image(str(FIGURES / "degree_distribution.png"))

## 8. Outlier indicators**These are indicators, not fraud labels.** This dataset has no manipulation groundtruth, so precision and recall cannot be computed. Quantitative validation happens onlyunder the synthetic-injection protocol in Phase 16, where labels exist by construction.

In [ ]:
outliers = json.loads((TABLES / "outlier_report.json").read_text())display(pd.Series(outliers["summary"]).to_frame("value"))print("Fitted on:", outliers["fitted_scope"], f"({outliers['mode']})")for note in outliers["notes"]:    print("-", note)

In [ ]:
scores = pd.read_csv(TABLES / "outlier_scores.csv", index_col=0)scores.nlargest(15, ["n_indicators", "iforest_score"])[    ["n_indicators", "iforest_score", "iqr_flags", "zscore_flags", "flag_reasons"]]